In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording
from pathlib import Path
import pickle
from utils_clique import (
    CliqueInfo,
    build_shank_cliques,
    neuron_inf_dict_to_dataframe,
    get_recording_clique,
    filter_neuron_inf_by_clique,
    filter_gt_detect_array_by_clique,
    prepare_training_data,
    train_autosort_model
)


/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 加载数据（与recordings_30channels_12_month.ipynb一致）
files = sorted(os.listdir("/media/ubuntu/sda/data/mouse6/ns4/natural_image"))
recording_list = []
for file in files:
    recording_raw = se.read_blackrock(file_path=f'/media/ubuntu/sda/data/mouse6/ns4/natural_image/{file}')
    recording_recorded = recording_raw.remove_channels(["98", '31', '32'])
    recording_list.append(recording_recorded.time_slice(start_time= 60, end_time = 1260))

probe_30channel = read_probeinterface('/media/ubuntu/sda/data/probe.json')
recording_recorded = si.concatenate_recordings(recording_list)
recording_recorded = recording_recorded.set_probegroup(probe_30channel)

recording_cmr = recording_recorded
recording_f = spre.bandpass_filter(recording_recorded, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_f, freq=60)

print(recording_f)
recording_cmr = spre.common_reference(recording_f, reference="global", operator="median")
recording_cmr = recording_cmr.rename_channels(['A-000', 'A-001', 'A-002', 'A-003', 'A-004',
                               'A-005', 'A-006', 'A-007', 'A-008', 'A-009',
                               'A-0010', 'A-011', 'A-012', 'A-013', 'A-014',
                               'A-015', 'A-016', 'A-017', 'A-018', 'A-019',
                               'A-020', 'A-021', 'A-022', 'A-023', 'A-024',
                               'A-025', 'A-026', 'A-027', 'A-028', 'A-029'])
print(recording_cmr)

# 设置输出文件夹（与recordings_30channels_12_month.ipynb中的output_folder一致）
output_folder = '/media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim'
combined_output_base = output_folder

# 计算session范围和session名称（与recordings_30channels_12_month.ipynb一致）
# 根据recording_list中每个recording的采样点数来计算session范围
# recording是通过concatenate_recordings合并的，合并后只有一个segment
# 需要根据原始recording_list来计算每个session的采样点范围
sampling_frequency = recording_cmr.get_sampling_frequency()

# 计算每个session的采样点范围
segment_sample_ranges = {}  # {segment_idx: (start_sample, end_sample)}
segment_num_samples_dict = {}  # {segment_idx: num_samples}
session_names = []  # 存储每个session的名称

# 获取每个session的文件名（去掉扩展名）
for i, file in enumerate(files):
    # 去掉文件扩展名，作为session名称
    session_name = Path(file).stem
    session_names.append(session_name)

# 根据recording_list中每个recording的采样点数计算范围
current_sample = 0
n_segments = len(recording_list)  # session数量等于recording_list的长度

for seg_idx in range(n_segments):
    # 获取该recording的采样点数（在合并前的原始recording）
    segment_num_samples = recording_list[seg_idx].get_num_samples()
    start_sample = current_sample
    end_sample = current_sample + segment_num_samples
    
    segment_sample_ranges[seg_idx] = (start_sample, end_sample)
    segment_num_samples_dict[seg_idx] = segment_num_samples
    
    session_name = session_names[seg_idx] if seg_idx < len(session_names) else f"session_{seg_idx}"
    print(f"Session {seg_idx} ({session_name}): 采样点范围 = [{start_sample}, {end_sample}), 采样点数 = {segment_num_samples}")
    
    current_sample = end_sample

print(f"\n共 {n_segments} 个sessions")



BandpassFilterRecording: 30 channels - 10000.0Hz - 1 segments - 192,000,000 samples 
                         19,200.00s (5.33 hours) - int16 dtype - 10.73 GiB
ChannelSliceRecording: 30 channels - 10000.0Hz - 1 segments - 192,000,000 samples 
                       19,200.00s (5.33 hours) - int16 dtype - 10.73 GiB
Session 0 (mouse6_012123_natural_image_001): 采样点范围 = [0, 12000000), 采样点数 = 12000000
Session 1 (mouse6_021322_natural_image_001): 采样点范围 = [12000000, 24000000), 采样点数 = 12000000
Session 2 (mouse6_022223_natural_image_001): 采样点范围 = [24000000, 36000000), 采样点数 = 12000000
Session 3 (mouse6_022522_natural_image_001): 采样点范围 = [36000000, 48000000), 采样点数 = 12000000
Session 4 (mouse6_031722_natural_image_001): 采样点范围 = [48000000, 60000000), 采样点数 = 12000000
Session 5 (mouse6_032123_natural_image_001): 采样点范围 = [60000000, 72000000), 采样点数 = 12000000
Session 6 (mouse6_042323_natural_image_001): 采样点范围 = [72000000, 84000000), 采样点数 = 12000000
Session 7 (mouse6_042422_natural_image_001): 采样点范围 = [

In [4]:
# Clique级别训练流程（适配30通道和session-based架构）
# 指定要训练的session
target_session_name = 'mouse6_021322_natural_image_001'

# 找到对应的session_idx
target_session_idx = None
for idx, name in enumerate(session_names):
    if name == target_session_name:
        target_session_idx = idx
        break

if target_session_idx is None:
    raise ValueError(f"未找到指定的session: {target_session_name}")

print(f"将训练session: {target_session_name} (index: {target_session_idx})")

# 创建单个包含所有30个通道的clique
probe = recording_cmr.get_probe()
probe_df = probe.to_dataframe()
all_channel_ids = probe_df['contact_ids'].astype(str).tolist()
cliques = [
    CliqueInfo(
        clique_id=0,
        device_channel_indices=list(range(len(all_channel_ids))),
        contact_ids=all_channel_ids,
        center=(probe_df['x'].mean(), probe_df['y'].mean())
    )
]

# 对每个clique和指定的session进行训练
for clique in cliques:
    clique_id = clique.clique_id
    print(f"\n{'='*60}")
    print(f"Processing Clique {clique_id}")
    print(f"{'='*60}")
    
    # 只处理指定的session
    session_idx = target_session_idx
    session_name = target_session_name
    print(f"\n处理 Session {session_idx} ({session_name})...")
    
    # 从新的文件架构读取数据
    session_data_folder = f'{combined_output_base}/clique_{clique_id}/{session_name}'
    neuron_inf_path = f'{session_data_folder}/neuron_inf.pickle'
    gt_detect_array_path = f'{session_data_folder}/gt_detect_array.csv'
    
    if not os.path.exists(neuron_inf_path) or not os.path.exists(gt_detect_array_path):
        print(f"  警告: {session_data_folder} 下没有找到数据文件，跳过")
        continue
    
    # 加载数据
    with open(neuron_inf_path, 'rb') as f:
        neuron_inf_dict = pickle.load(f)
    gt_detect_array = pd.read_csv(gt_detect_array_path)
    
    # 转换为DataFrame
    neuron_inf_session = neuron_inf_dict_to_dataframe(neuron_inf_dict)
    
    print(f"  Neurons: {len(neuron_inf_session)}")
    print(f"  Spikes: {len(gt_detect_array)}")
    
    # 从recording_cmr中提取该session的recording（根据采样点范围）
    # gt_detect_array的时间是session内的相对时间，需要从recording_cmr中提取对应的segment
    start_sample, end_sample = segment_sample_ranges[session_idx]
    
    # 从recording_cmr中提取该session的recording
    session_recording = recording_cmr.frame_slice(start_frame=start_sample, end_frame=end_sample)
    
    # 获取recording_clique（对于30通道，clique包含所有通道，所以recording_clique就是session_recording）
    recording_clique = get_recording_clique(session_recording, clique)
    print(f"  Recording clique channels: {len(recording_clique.get_channel_ids())}")
    
    # 准备训练数据
    clique_save_dir = f'{combined_output_base}/clique_{clique_id}/{session_name}'
    train_data_dir = prepare_training_data(
        recording_f=recording_clique,
        gt_detect_array=gt_detect_array,
        neuron_inf=neuron_inf_session,
        save_dir=clique_save_dir,
        duration_seconds=300,
        thr_min=2.5,
        thr_max=10,
        distance=3,
        wlen=5,
        prominence=15,
        left_sample=10,
        right_sample=20,
        max_firing_channel=None
    )
    
    # 训练模型（重复5次）
    n_channels = recording_clique.get_num_channels()
    n_repeats = 5
    
    for repeat_idx in range(1, n_repeats + 1):
        print(f"\n  ===== 重复训练 {repeat_idx}/{n_repeats} =====")
        model_save_dir = f'{clique_save_dir}/model_{repeat_idx}'
        
        autosort_model, training_log = train_autosort_model(
            train_data_dir=train_data_dir,
            model_save_dir=model_save_dir,
            n_channels=n_channels,
            left_sample=10,
            right_sample=20,
            epochs=20,
            batch_size=512,
            device=None,
            early_stopping=True,
            patience=5,
            min_delta=0.0,
            use_focal_loss=True,
            focal_gamma=2.0
        )
        
        print(f"  重复训练 {repeat_idx}/{n_repeats} 完成!")
    
    print(f"  Clique {clique_id}, Session {session_name} 所有重复训练完成!")

print("\n所有训练完成！")


将训练session: mouse6_021322_natural_image_001 (index: 1)

Processing Clique 0

处理 Session 1 (mouse6_021322_natural_image_001)...
  Neurons: 34
  Spikes: 268116
  Recording clique channels: 30
### 1. Threshold Detection
Sampling rate: 10000.0 Hz, Number of channels: 30
Recording total length: 12000000 samples (1200.00 seconds)
Will process first 3000000 samples (300.00 seconds)
Data shape: (3000000, 30) (clique channels)
Using old detection method: extremum_channels
Using 26 valid channels from neuron extremum_channels
Building detect_array...
Number of detected spikes: 1102239
去重: 移除了130776个spikes（保留幅值更大的channel上的spike）
去重前: 1102239个spikes, 去重后: 971463个spikes

### 2. Load Ground Truth and Match
Building gt_array from gt_detect_array...
Filtered gt_detect_array: 67639 spikes (out of 268116 total)
GT spikes time range: 18 - 2999925 (sample points)
Expected time range: 0 - 3000000 (sample points)
Recording clique channel IDs (keys in probe_to_clique_index): ['A-000', 'A-001', 'A-002', 'A-00

Extracting waveforms:  17%|█▋        | 5/30 [00:02<00:10,  2.45it/s]


KeyboardInterrupt: 

In [8]:
# ============================================================
# 绘制每个clique的特征UMAP图
# ============================================================
# 每个clique生成一个PDF，包含4张UMAP图：
# 1. Noise detection GT
# 2. Noise detection predicted
# 3. Label classifier GT
# 4. Label classifier predicted

from umap import UMAP
import torch
from torch.utils import data
from tqdm import tqdm
from matplotlib.backends.backend_pdf import PdfPages

print("="*60)
print("绘制每个clique的特征UMAP图")
print("="*60)

# 重新导入utils_clique以确保使用最新的代码定义
import importlib
import utils_clique
importlib.reload(utils_clique)
SimpleAutoSort = utils_clique.SimpleAutoSort
SimpleWaveformLoader = utils_clique.SimpleWaveformLoader

# 指定要处理的session（与训练时一致）
target_session_name = 'mouse6_021322_natural_image_001'

# 找到对应的session_idx
target_session_idx = None
for idx, name in enumerate(session_names):
    if name == target_session_name:
        target_session_idx = idx
        break

if target_session_idx is None:
    raise ValueError(f"未找到指定的session: {target_session_name}")

print(f"将处理session: {target_session_name} (index: {target_session_idx})")

for clique in cliques:
    clique_id = clique.clique_id
    print(f"\n{'='*60}")
    print(f"处理Clique {clique_id}")
    print(f"{'='*60}")
    
    # 只处理指定的session
    session_idx = target_session_idx
    session_name = target_session_name
    print(f"\n处理 Session {session_idx} ({session_name})...")
    
    session_data_folder = f'{combined_output_base}/clique_{clique_id}/{session_name}'
    train_data_dir = f'{session_data_folder}/train_data/'
    
    # 检查文件是否存在
    if not os.path.exists(train_data_dir):
        print(f"  警告: {train_data_dir} 不存在，跳过")
        continue
    
    # 尝试从model_1加载classification_mapping（如果不存在，尝试其他model）
    model_save_dir = None
    classification_mapping_path = None
    for repeat_idx in range(1, 6):  # 尝试model_1到model_5
        candidate_model_dir = f'{session_data_folder}/model_{repeat_idx}'
        candidate_mapping_path = f'{candidate_model_dir}/classification_mapping.pkl'
        if os.path.exists(candidate_mapping_path):
            model_save_dir = candidate_model_dir
            classification_mapping_path = candidate_mapping_path
            break
    
    if classification_mapping_path is None or not os.path.exists(classification_mapping_path):
        print(f"  警告: 未找到classification_mapping.pkl，跳过")
        continue
    
    with open(classification_mapping_path, 'rb') as f:
        classification_mapping = pickle.load(f)
    keep_id_list = classification_mapping['label_list']
    
    n_channels = 30  # 30通道
    samplepoints = 30
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 创建模型
    autosort_model = SimpleAutoSort(
        ch_num=n_channels,
        samplepoints=samplepoints,
        device=device,
        set_shank_id=keep_id_list,
        save_dir=model_save_dir,
        pos_weight_noise=None,
        pos_weight_label=None
    )
    
    # 加载模型权重
    noise_model_path = f'{model_save_dir}/multitask_single_wave_clsfier_noise_clsfier.pth'
    label_model_path = f'{model_save_dir}/multitask_single_wave_clsfier_label_clsfier.pth'
    
    if not os.path.exists(noise_model_path) or not os.path.exists(label_model_path):
        print(f"  警告: 模型文件不存在，跳过")
        continue
    
    autosort_model.clsfier_noise.load_state_dict(torch.load(noise_model_path, map_location=device))
    autosort_model.clsfier_label.load_state_dict(torch.load(label_model_path, map_location=device))
    autosort_model.eval()
    
    print(f"  模型已加载")
    
    # 加载训练数据
    # 需要知道shank_channel，这里使用所有通道（0到29，共30个通道）
    shank_channel = list(range(n_channels))
    dataset = SimpleWaveformLoader(train_data_dir, shank_channel, Keep_id=keep_id_list)
    dataloader = data.DataLoader(dataset, batch_size=512, shuffle=False, num_workers=0)
    
    print(f"  数据集大小: {len(dataset)}")
    
    # 提取特征和预测
    all_noise_features = []  # Features for noise classifier (intermediate_forward)
    all_label_features = []  # Features for label classifier (intermediate_forward)
    all_noise_gt = []
    all_noise_pred = []
    all_label_gt = []
    all_label_pred = []
    
    print(f"  提取特征和预测...")
    with torch.no_grad():
        for batch_data in tqdm(dataloader, desc=f"Processing batches"):
                # SimpleWaveformLoader返回顺序: Img (n_channels, window_length), GT (unit one-hot), GT_binary (noise one-hot), Img_single, channel_index
                batch_Img, batch_unit_label_onehot, batch_noise_label_onehot, batch_single, batch_channel_indices = batch_data
                batch_Img = batch_Img.to(device)  # (batch_size, n_channels, window_length)
                batch_single = batch_single.to(device)  # (batch_size, window_length)
                batch_noise_label_onehot = batch_noise_label_onehot.to(device)
                batch_unit_label_onehot = batch_unit_label_onehot.to(device)
                batch_channel_indices = batch_channel_indices.to(device) if isinstance(batch_channel_indices, torch.Tensor) else torch.tensor(batch_channel_indices, device=device)
                
                # Flatten batch_Img to (batch_size, n_channels * window_length) for _prepare_input
                batch_size = batch_Img.shape[0]
                batch_multi = batch_Img.view(batch_size, -1)  # (batch_size, n_channels * window_length)
                
                # Prepare input: codes will be (batch_size, n_channels + 2, window_length)
                codes = autosort_model._prepare_input(batch_multi, batch_single, batch_channel_indices)
                
                # Noise classifier
                noise_features = autosort_model.clsfier_noise.intermediate_forward(codes)
                noise_output = autosort_model.clsfier_noise(codes)
                noise_pred = torch.argmax(noise_output, dim=1)  # (batch_size,)
                
                # Label classifier
                label_features = autosort_model.clsfier_label.intermediate_forward(codes)
                label_output = autosort_model.clsfier_label(codes)
                label_pred = torch.argmax(label_output, dim=1)  # (batch_size,)
                
                # 将one-hot标签转换为类别索引
                # batch_noise_label_onehot: (batch_size, 2) [noise, spike] -> 0=noise, 1=spike
                noise_gt = torch.argmax(batch_noise_label_onehot, dim=1)  # (batch_size,)
                # batch_unit_label_onehot: (batch_size, n_units) -> label index (如果全为0则label=-1表示noise)
                unit_label_gt = torch.argmax(batch_unit_label_onehot, dim=1)  # (batch_size,)
                # 如果one-hot全为0（即不在任何unit中），则argmax会返回0，需要检查是否真的是valid unit
                # 可以通过检查max值来判断：如果max值为0，则表示不是valid unit
                unit_label_valid = torch.max(batch_unit_label_onehot, dim=1)[0] > 0  # (batch_size,)
                unit_label_gt = torch.where(unit_label_valid, unit_label_gt, torch.tensor(-1, device=device))
                
                # 保存特征和标签
                all_noise_features.append(noise_features.cpu().numpy())
                all_label_features.append(label_features.cpu().numpy())
                all_noise_gt.append(noise_gt.cpu().numpy())
                all_noise_pred.append(noise_pred.cpu().numpy())
                all_label_gt.append(unit_label_gt.cpu().numpy())
                all_label_pred.append(label_pred.cpu().numpy())
        
        # 合并所有batch
        all_noise_features = np.concatenate(all_noise_features, axis=0)  # (n_samples, 30)
        all_label_features = np.concatenate(all_label_features, axis=0)  # (n_samples, 30)
        all_noise_gt = np.concatenate(all_noise_gt, axis=0)  # (n_samples,)
        all_noise_pred = np.concatenate(all_noise_pred, axis=0)  # (n_samples,)
        all_label_gt = np.concatenate(all_label_gt, axis=0)  # (n_samples,)
        all_label_pred = np.concatenate(all_label_pred, axis=0)  # (n_samples,)
        
        print(f"  特征提取完成:")
        print(f"    - Noise features shape: {all_noise_features.shape}")
        print(f"    - Label features shape: {all_label_features.shape}")
        print(f"    - Noise GT: {np.unique(all_noise_gt)}")
        print(f"    - Noise Pred: {np.unique(all_noise_pred)}")
        print(f"    - Label GT unique count: {len(np.unique(all_label_gt[all_label_gt >= 0]))}")
        print(f"    - Label Pred unique count: {len(np.unique(all_label_pred))}")
        
        # 限制样本数量以提高UMAP计算速度（如果数据太多）
        # 对于noise的UMAP，从全部数据中采样
        max_samples_for_noise_umap = 50000
        if len(all_noise_features) > max_samples_for_noise_umap:
            print(f"  数据量较大，随机采样 {max_samples_for_noise_umap} 个样本用于Noise UMAP")
            noise_indices = np.random.choice(len(all_noise_features), max_samples_for_noise_umap, replace=False)
            noise_features_for_umap = all_noise_features[noise_indices]
            noise_gt_for_umap = all_noise_gt[noise_indices]
            noise_pred_for_umap = all_noise_pred[noise_indices]
        else:
            noise_features_for_umap = all_noise_features
            noise_gt_for_umap = all_noise_gt
            noise_pred_for_umap = all_noise_pred
        
        # 对于label的UMAP，先从全部数据中筛选出spike（label >= 0），然后采样5000个点
        max_samples_for_label_umap = 30000
        spike_mask = all_label_gt >= 0  # 筛选出spike
        spike_indices = np.where(spike_mask)[0]
        
        if len(spike_indices) > max_samples_for_label_umap:
            print(f"  从 {len(spike_indices)} 个spike中随机采样 {max_samples_for_label_umap} 个样本用于Label UMAP")
            selected_spike_indices = np.random.choice(len(spike_indices), max_samples_for_label_umap, replace=False)
            label_indices = spike_indices[selected_spike_indices]
        else:
            print(f"  使用全部 {len(spike_indices)} 个spike用于Label UMAP")
            label_indices = spike_indices
        
        label_features_for_umap = all_label_features[label_indices]
        label_gt_for_umap = all_label_gt[label_indices]
        label_pred_for_umap = all_label_pred[label_indices]
        
        # 同时需要对应的noise预测结果，用于绘制label predicted图
        noise_pred_for_label_umap = all_noise_pred[label_indices]
        
        # UMAP降维
        print(f"  进行UMAP降维...")
        umap_noise = UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
        umap_label = UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
        
        noise_features_2d = umap_noise.fit_transform(noise_features_for_umap)
        label_features_2d = umap_label.fit_transform(label_features_for_umap)
        
        # 保存PDF
        pdf_path = f'{model_save_dir}/umap_visualization_clique_{clique_id}_{session_name}.pdf'
        print(f"  保存UMAP图到: {pdf_path}")
        
        with PdfPages(pdf_path) as pdf:
            # 1. Noise detection GT
            fig, ax = plt.subplots(1, 1, figsize=(6, 6))
            unique_labels = sorted(np.unique(noise_gt_for_umap))
            colors = ['lightgrey', 'orange']
            label_names = ['Noise', 'Spike']  # 通常0=noise, 1=spike
            
            for i, label in enumerate(unique_labels):
                mask = noise_gt_for_umap == label
                label_name = label_names[int(label)] if int(label) < len(label_names) else f'Class {int(label)}'
                ax.scatter(noise_features_2d[mask, 0], noise_features_2d[mask, 1], 
                          c=[colors[i]], label=label_name, alpha=1, s=1)
            
            ax.axis('off')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
            
            # 2. Noise detection Predicted
            fig, ax = plt.subplots(1, 1, figsize=(6, 6))
            unique_labels = sorted(np.unique(noise_pred_for_umap))
            colors = ['lightgrey', 'orange']
            
            for i, label in enumerate(unique_labels):
                mask = noise_pred_for_umap == label
                label_name = label_names[int(label)] if int(label) < len(label_names) else f'Class {int(label)}'
                ax.scatter(noise_features_2d[mask, 0], noise_features_2d[mask, 1], 
                          c=[colors[i]], label=label_name, alpha=1, s=1)
            
            ax.axis('off')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
            
            fig, ax = plt.subplots(1, 1, figsize=(6, 6))
            if len(label_gt_for_umap) > 0:
                valid_features = label_features_2d
                valid_labels = label_gt_for_umap
                
                unique_labels = sorted(np.unique(valid_labels))
                colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))
                
                for i, label in enumerate(unique_labels):
                    mask = valid_labels == label
                    # 将label索引映射回unit ID
                    if int(label) < len(keep_id_list):
                        unit_id = keep_id_list[int(label)]
                        label_name = f'Unit {unit_id}'
                    else:
                        label_name = f'Label {int(label)}'
                    ax.scatter(valid_features[mask, 0], valid_features[mask, 1], 
                              c=[colors[i]], label=label_name, alpha=1, s=1)
            
            ax.axis('off')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
            
            fig, ax = plt.subplots(1, 1, figsize=(6, 6))
            if len(label_pred_for_umap) > 0:
                valid_features = label_features_2d
                valid_labels = label_pred_for_umap
                
                unique_labels = sorted(np.unique(valid_labels))
                colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))
                
                for i, label in enumerate(unique_labels):
                    mask = valid_labels == label
                    # 将label索引映射回unit ID
                    if int(label) < len(keep_id_list):
                        unit_id = keep_id_list[int(label)]
                        label_name = f'Unit {unit_id}'
                    else:
                        label_name = f'Label {int(label)}'
                    ax.scatter(valid_features[mask, 0], valid_features[mask, 1], 
                              c=[colors[i]], label=label_name, alpha=1, s=1)
            ax.axis('off')
            plt.tight_layout()
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
        
        print(f"  PDF已保存: {pdf_path}")

print(f"\n{'='*60}")
print("所有UMAP图绘制完成")
print(f"{'='*60}")


绘制每个clique的特征UMAP图
将处理session: mouse6_021322_natural_image_001 (index: 1)

处理Clique 0

处理 Session 1 (mouse6_021322_natural_image_001)...
  模型已加载
Dataset loaded:
  - Total samples: 3241864
  - Number of channels: 30
  - Window length: 30
  - Number of unique units: 34
  - Noise samples: 3065886.0
  - Non-noise samples: 175978.0
  数据集大小: 3241864
  提取特征和预测...


Processing batches: 100%|██████████| 6332/6332 [00:22<00:00, 277.90it/s]


  特征提取完成:
    - Noise features shape: (3241864, 30)
    - Label features shape: (3241864, 30)
    - Noise GT: [0 1]
    - Noise Pred: [0 1]
    - Label GT unique count: 34
    - Label Pred unique count: 34
  数据量较大，随机采样 50000 个样本用于Noise UMAP
  从 175978 个spike中随机采样 30000 个样本用于Label UMAP
  进行UMAP降维...
  保存UMAP图到: /media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim/clique_0/mouse6_021322_natural_image_001/model_1/umap_visualization_clique_0_mouse6_021322_natural_image_001.pdf
  PDF已保存: /media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim/clique_0/mouse6_021322_natural_image_001/model_1/umap_visualization_clique_0_mouse6_021322_natural_image_001.pdf

所有UMAP图绘制完成
